# NB_FIGS — Q1 figures from your saved Drive results

Generates every figure for the crossover paper **from the real result CSVs on
Drive** — nothing is hard-coded. It also prints the LaTeX macro values so the
figures and the manuscript numbers stay in sync.

**Run order:** Cell 1 (mount + locate files) → Cell 2 (load) → Cells 3–8 (figures).
Re-runnable; reads only, never modifies your results.


## Cell 1 — mount Drive and locate the result files

In [1]:
from google.colab import drive; drive.mount("/content/drive")
import os, glob, math, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy import stats
plt.rcParams.update({"font.size":10,"font.family":"serif","axes.spines.top":False,
    "axes.spines.right":False,"figure.dpi":130,"savefig.bbox":"tight"})

ROOT = "/content/drive/MyDrive/nb4_outputs"
FIG  = os.path.join(ROOT,"figures_paper"); os.makedirs(FIG,exist_ok=True)
CKPT = os.path.join(ROOT,"checkpoints")        # NB4 three-arm per-market *_rl.csv
EXT  = os.path.join(ROOT,"ext_checkpoints")    # NB5 per-market *_armA.csv/_armB.csv
BLUE,RED,GREY,GREEN="#2c6fbb","#c0392b","#7f8c8d","#27ae60"

def _find(*names):
    for n in names:
        for base in [ROOT, CKPT, EXT, "."]:
            p=os.path.join(base,n)
            if os.path.exists(p): return p
    return None

def savefig(fig,name):
    fig.savefig(os.path.join(FIG,name+".pdf")); fig.savefig(os.path.join(FIG,name+".png"),dpi=300)
    plt.close(fig); print("  saved",name)

def pick(df, *opts):
    low={c.lower().strip():c for c in df.columns}
    for o in opts:
        if o.lower() in low: return low[o.lower()]
    return None

def show_cols(path):
    try:
        import pandas as pd; d=pd.read_csv(path, nrows=3)
        print("   cols:", list(d.columns))
    except Exception as e: print("   (could not read:", e, ")")

print("figures ->",FIG)
for label,pat in [("NB4 three-arm per-market", os.path.join(CKPT,"*_rl.csv")),
                  ("NB4 combined","risk_rl_results.csv"),
                  ("NB5 armA per-market", os.path.join(EXT,"*_armA.csv")),
                  ("NB5 armB per-market", os.path.join(EXT,"*_armB.csv")),
                  ("NB5 combined","ext_results.csv"),
                  ("vol forecast (manip.)","vol_forecast_results.csv")]:
    hits = glob.glob(pat) if "*" in pat else ([_find(pat)] if _find(pat) else [])
    print(f"  {label:26s}: {len(hits)} file(s)")
    if hits and hits[0]: show_cols(hits[0])

Mounted at /content/drive
figures -> /content/drive/MyDrive/nb4_outputs/figures_paper
  NB4 three-arm per-market  : 14 file(s)
   cols: ['market', 'group', 'agent', 'state', 'seed', 'NetWorth', 'Sharpe', 'Sortino', 'MaxDD', 'CAGR', 'Calmar']
  NB4 combined              : 1 file(s)
   cols: ['market', 'group', 'agent', 'state', 'seed', 'NetWorth', 'Sharpe', 'Sortino', 'MaxDD', 'CAGR', 'Calmar']
  NB5 armA per-market       : 14 file(s)
   cols: ['market', 'group', 'arm', 'agent', 'state', 'seed', 'NetWorth', 'Sharpe', 'Sortino', 'MaxDD', 'CAGR', 'Calmar']
  NB5 armB per-market       : 14 file(s)
   cols: ['market', 'group', 'arm', 'agent', 'state', 'seed', 'NetWorth', 'Sharpe', 'Sortino', 'MaxDD', 'CAGR', 'Calmar']
  NB5 combined              : 1 file(s)
   cols: ['market', 'group', 'arm', 'agent', 'state', 'seed', 'NetWorth', 'Sharpe', 'Sortino', 'MaxDD', 'CAGR', 'Calmar']
  vol forecast (manip.)     : 1 file(s)
   cols: ['market', 'model', 'RMSE', 'DM_vs_RW', 'DM_p']


## Cell 2 — load NB4 (profit reward) and NB5 (DSR reward) into two frames

`load_threearm` accepts either the per-market `*_rl.csv` files or a combined
`risk_rl_results.csv`, and keeps only files that actually contain the three-arm
design (Baseline / +RetFore / +VolFore).

In [2]:
def _read_many(paths):
    fr=[pd.read_csv(p) for p in paths if os.path.exists(p)]
    return pd.concat(fr,ignore_index=True) if fr else None

def load_threearm():
    comb=_find("risk_rl_results.csv")
    d = pd.read_csv(comb) if comb else _read_many(glob.glob(os.path.join(CKPT,"*_rl.csv")))
    if d is None: return None
    d.columns=[c.strip() for c in d.columns]
    if "sharpe" in d.columns and "Sharpe" not in d.columns: d=d.rename(columns={"sharpe":"Sharpe"})
    if "+VolFore" not in set(d.get("state",[])): return None      # not the three-arm file
    return d

def load_dsr(arm="A"):
    comb=_find("ext_results.csv")
    if comb:
        d=pd.read_csv(comb)
        if "arm" in d.columns: d=d[d.arm==arm]
    else:
        d=_read_many(glob.glob(os.path.join(EXT,f"*_arm{arm}.csv")))
    if d is not None: d.columns=[c.strip() for c in d.columns]
    return d

NB4 = load_threearm()      # profit reward, 3 arms, 14 markets
NB5A= load_dsr("A")        # DSR reward, 3 arms
NB5B= load_dsr("B")        # DSR reward, residual vs VT rule
for nm,df in [("NB4 (profit)",NB4),("NB5 armA (DSR)",NB5A),("NB5 armB (DSR)",NB5B)]:
    if df is None: print(nm,": MISSING"); continue
    mk=sorted(df.market.unique()); print(f"{nm}: {len(df)} rows, {len(mk)} markets -> {mk}")

def pooled_paired(df, a, b, metric="Sharpe"):
    r=df[df.seed>=0]
    piv=r.pivot_table(index=["market","agent","seed"],columns="state",values=metric).dropna(subset=[a,b])
    d=piv[a]-piv[b]
    try: _,p=stats.wilcoxon(d,alternative="greater")
    except ValueError: p=float("nan")
    return d.mean(), d.median(), p, len(d)

NB4 (profit): 854 rows, 14 markets -> ['ASX200', 'BTC', 'Bovespa', 'DAX', 'DSE', 'ETH', 'FTSE100', 'KOSPI', 'MexIPC', 'NASDAQ', 'Nifty50', 'Nikkei', 'SP500', 'Vietnam']
NB5 armA (DSR): 434 rows, 14 markets -> ['ASX200', 'BTC', 'Bovespa', 'DAX', 'DSE', 'ETH', 'FTSE100', 'KOSPI', 'MexIPC', 'NASDAQ', 'Nifty50', 'Nikkei', 'SP500', 'Vietnam']
NB5 armB (DSR): 285 rows, 14 markets -> ['ASX200', 'BTC', 'Bovespa', 'DAX', 'DSE', 'ETH', 'FTSE100', 'KOSPI', 'MexIPC', 'NASDAQ', 'Nifty50', 'Nikkei', 'SP500', 'Vietnam']


## Cell 3 — Fig 0: experimental-design schematic (diagram, not data)

In [3]:
fig,ax=plt.subplots(figsize=(7.2,3.4)); ax.axis("off"); ax.set_xlim(0,10); ax.set_ylim(0,6)
def box(x,y,w,h,t,fc="#eef3fb",ec=BLUE):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.04,rounding_size=0.12",fc=fc,ec=ec,lw=1.3))
    ax.text(x+w/2,y+h/2,t,ha="center",va="center",fontsize=9)
box(0.2,2.5,1.7,1.0,"14 markets\n4 agents\n5 seeds",fc="#f4f4f4",ec=GREY)
for i,(ry,rc,rl) in enumerate([(4.1,BLUE,"Reward = Profit"),(0.7,RED,"Reward = DSR")]):
    box(2.5,ry-0.15,1.9,1.3,rl,fc=("#eef3fb" if i==0 else "#fbeeee"),ec=rc)
    for j,(arm,ac) in enumerate([("Baseline",GREY),("+RetFore\n(control)",GREEN),("+VolFore\n(treatment)",rc)]):
        box(5.0+j*1.65,ry-0.15,1.5,1.3,arm,ec=ac)
    ax.add_patch(FancyArrowPatch((4.4,ry+0.5),(5.0,ry+0.5),arrowstyle="->",mutation_scale=12,color=rc))
ax.add_patch(FancyArrowPatch((1.9,3.0),(2.5,4.6),arrowstyle="->",mutation_scale=12,color=GREY))
ax.add_patch(FancyArrowPatch((1.9,3.0),(2.5,1.2),arrowstyle="->",mutation_scale=12,color=GREY))
ax.text(7.3,5.75,"same data / agents / seeds / costs — only the reward differs",ha="center",fontsize=8.5,style="italic",color="#444")
ax.text(9.9,3.0,"reward $\\times$ signal\ninteraction\n(DiD)",ha="right",va="center",fontsize=8.5,color="#333",
        bbox=dict(boxstyle="round",fc="#fff8e1",ec="#e0b000"))
savefig(fig,"fig0_design")

  saved fig0_design


## Cell 4 — Fig 1: the crossover (paper's central figure)

Computed on the markets present under **both** rewards (matched sample). Prints
the LaTeX macros to paste into `main.tex`.

In [4]:
if NB4 is None or NB5A is None:
    print("skip fig1 — need NB4 (profit) and NB5 armA (DSR); see Cell 1 listing"); raise SystemExit
common=sorted(set(NB4.market)&set(NB5A.market))
print("matched markets:",common)
def contrasts(df):
    sub=df[df.market.isin(common)]
    v=pooled_paired(sub,"+VolFore","Baseline"); r=pooled_paired(sub,"+RetFore","Baseline")
    return v,r
(vP,rP),(vD,rD)=contrasts(NB4),contrasts(NB5A)
print(f"PROFIT  VolFore-Base {vP[0]:+.3f} p={vP[2]:.3f} | RetFore-Base {rP[0]:+.3f} p={rP[2]:.3f}")
print(f"DSR     VolFore-Base {vD[0]:+.3f} p={vD[2]:.3f} | RetFore-Base {rD[0]:+.3f} p={rD[2]:.3f}")

x=[0,1]; fig,ax=plt.subplots(figsize=(5.6,4.3)); ax.axhline(0,color=GREY,lw=0.8,ls="--")
ax.plot(x,[vP[0],vD[0]],"-o",color=BLUE,lw=2.4,ms=9,label="+VolFore $-$ Baseline",zorder=3)
ax.plot(x,[rP[0],rD[0]],"-s",color=GREEN,lw=2.4,ms=9,label="+RetFore $-$ Baseline",zorder=3)
def lab(xi,m,p,c,dx,ha): ax.annotate(f"{m:+.3f}  ($p$={p:.2g})",(xi,m),textcoords="offset points",xytext=(dx,0),ha=ha,va="center",fontsize=8.5,color=c)
lab(0,vP[0],vP[2],BLUE,-10,"right"); lab(1,vD[0],vD[2],BLUE,12,"left")
lab(0,rP[0],rP[2],GREEN,-10,"right"); lab(1,rD[0],rD[2],GREEN,12,"left")
ax.set_xticks(x); ax.set_xticklabels(["Profit reward","DSR reward"],fontsize=11)
ax.set_ylabel("pooled paired $\\Delta$Sharpe (out-of-sample)")
ax.set_xlim(-0.75,1.75); ax.set_title("The value of a signal flips with the reward",fontsize=11.5,pad=10)
ax.legend(frameon=False,fontsize=9,loc="upper center",bbox_to_anchor=(0.5,-0.11),ncol=2)
savefig(fig,"fig1_crossover")

print("\n% ==== paste into main.tex DSR RESULT MACROS (matched sample) ====")
print(f"\\newcommand{{\\xoverProfVol}}{{{vP[0]:+.3f}}}\\newcommand{{\\xoverProfVolP}}{{{vP[2]:.3f}}}")
print(f"\\newcommand{{\\xoverProfRet}}{{{rP[0]:+.3f}}}\\newcommand{{\\xoverProfRetP}}{{{rP[2]:.2g}}}")
print(f"\\newcommand{{\\xoverDsrVol}}{{{vD[0]:+.3f}}}\\newcommand{{\\xoverDsrVolP}}{{{vD[2]:.2g}}}")
print(f"\\newcommand{{\\xoverDsrRet}}{{{rD[0]:+.3f}}}\\newcommand{{\\xoverDsrRetP}}{{{rD[2]:.3f}}}")

matched markets: ['ASX200', 'BTC', 'Bovespa', 'DAX', 'DSE', 'ETH', 'FTSE100', 'KOSPI', 'MexIPC', 'NASDAQ', 'Nifty50', 'Nikkei', 'SP500', 'Vietnam']
PROFIT  VolFore-Base +0.121 p=0.001 | RetFore-Base -0.077 p=0.980
DSR     VolFore-Base +0.081 p=0.082 | RetFore-Base -0.075 p=0.988
  saved fig1_crossover

% ==== paste into main.tex DSR RESULT MACROS (matched sample) ====
\newcommand{\xoverProfVol}{+0.121}\newcommand{\xoverProfVolP}{0.001}
\newcommand{\xoverProfRet}{-0.077}\newcommand{\xoverProfRetP}{0.98}
\newcommand{\xoverDsrVol}{+0.081}\newcommand{\xoverDsrVolP}{0.082}
\newcommand{\xoverDsrRet}{-0.075}\newcommand{\xoverDsrRetP}{0.988}


## Cell 5 — Fig 2: per-market volatility effect under the profit reward (forest)

In [5]:
if NB4 is None:
    print("skip fig2 — need NB4 three-arm results"); raise SystemExit
r=NB4[NB4.seed>=0]
piv=r.pivot_table(index=["market","agent","seed"],columns="state",values="Sharpe").dropna(subset=["+VolFore","Baseline"])
eff=(piv["+VolFore"]-piv["Baseline"]).groupby(level="market")
m=eff.mean().sort_values(); se=eff.std()/eff.count()**0.5
fig,ax=plt.subplots(figsize=(5.2,max(3.2,0.32*len(m)))); y=np.arange(len(m))
ax.errorbar(m.values,y,xerr=1.96*se.reindex(m.index).values,fmt="o",color=BLUE,ecolor=GREY,capsize=3)
ax.axvline(0,color=GREY,ls="--",lw=0.8); ax.axvline(m.mean(),color=RED,ls=":",lw=1.2,label=f"pooled mean {m.mean():+.3f}")
ax.set_yticks(y); ax.set_yticklabels(m.index,fontsize=8); ax.set_xlabel("+VolFore $-$ Baseline ($\\Delta$Sharpe)")
ax.set_title(f"Per-market volatility effect, profit reward ({len(m)} markets)",fontsize=10); ax.legend(frameon=False,fontsize=8)
savefig(fig,"fig2_forest")
vv=pooled_paired(NB4,"+VolFore","Baseline"); rr=pooled_paired(NB4,"+RetFore","Baseline"); cc=pooled_paired(NB4,"+VolFore","+RetFore")
print("\n% profit-reward pooled (all NB4 markets):")
print(f"  VolFore-Base {vv[0]:+.3f} p={vv[2]:.4f} n={vv[3]}")
print(f"  RetFore-Base {rr[0]:+.3f} p={rr[2]:.4f}")
print(f"  VolFore-RetFore {cc[0]:+.3f} p={cc[2]:.4f}")

  saved fig2_forest

% profit-reward pooled (all NB4 markets):
  VolFore-Base +0.121 p=0.0009 n=280
  RetFore-Base -0.077 p=0.9805
  VolFore-RetFore +0.198 p=0.0000


## Cell 6 — Fig 3: forecast accuracy vs economic value

Needs `vol_forecast_results.csv` (HAR vs random-walk-vol). Skips cleanly if absent.

In [6]:
p=_find("vol_forecast_results.csv")
if p is None or NB4 is None:
    print("skip fig3 — need vol_forecast_results.csv + NB4")
else:
    vf=pd.read_csv(p); vf.columns=[c.strip() for c in vf.columns]
    dmc=pick(vf,"DM_stat","DM_vs_RW","DM","dm_stat","DMstat")
    rmc=pick(vf,"RMSE","rmse","val_RMSE","test_RMSE")
    if dmc is None or rmc is None:
        print("skip fig3 — vol_forecast file lacks DM/RMSE cols; found:",list(vf.columns)); raise SystemExit
    q=(vf.loc[vf.groupby("market")[rmc].idxmin()].set_index("market")[dmc])*-1  # larger=better
    r=NB4[NB4.seed>=0]; piv=r.pivot_table(index=["market","agent","seed"],columns="state",values="Sharpe").dropna(subset=["+VolFore","Baseline"])
    ben=(piv["+VolFore"]-piv["Baseline"]).groupby(level="market").mean()
    idx=[m for m in q.index if m in ben.index]; xx=q.reindex(idx).values; yy=ben.reindex(idx).values
    rho,pp=stats.spearmanr(xx,yy)
    fig,ax=plt.subplots(figsize=(5.0,4.0)); ax.scatter(xx,yy,color=BLUE)
    for i,mk in enumerate(idx): ax.annotate(mk,(xx[i],yy[i]),fontsize=7,xytext=(3,3),textcoords="offset points")
    ax.axhline(0,color=GREY,ls="--",lw=0.8); ax.set_xlabel("forecast quality (−DM stat, larger=better)")
    ax.set_ylabel("trading benefit +VolFore−Baseline")
    ax.set_title(f"Accuracy does not predict value ($\\rho$={rho:.2f}, p={pp:.2f})",fontsize=10)
    savefig(fig,"fig3_accuracy_value")
    pc=pick(vf,'DM_p','DM_pvalue','p','pval')
    nsig=(vf.loc[vf.groupby('market')[rmc].idxmin(),pc]<0.05).sum() if pc else float('nan')
    print(f"\nmanipulation check: HAR beats RW-vol (DM p<0.05) in {nsig}/{vf.market.nunique()} markets")

  saved fig3_accuracy_value

manipulation check: HAR beats RW-vol (DM p<0.05) in 14/14 markets


## Cell 7 — Fig 4: manipulation check bar (HAR vs random-walk volatility)

In [7]:
p=_find("vol_forecast_results.csv")
if p is None:
    print("skip fig4 — need vol_forecast_results.csv")
else:
    vf=pd.read_csv(p); vf.columns=[c.strip() for c in vf.columns]
    dmc=pick(vf,"DM_stat","DM_vs_RW","DM","dm_stat","DMstat")
    rmc=pick(vf,"RMSE","rmse","val_RMSE","test_RMSE"); pc=pick(vf,"DM_p","DM_pvalue","p","pval")
    if dmc is None or rmc is None or pc is None:
        print("skip fig4 — need DM/RMSE/p cols; found:",list(vf.columns)); raise SystemExit
    best=vf.loc[vf.groupby("market")[rmc].idxmin()].set_index("market")
    dm=(-best[dmc]).sort_values()               # larger = HAR better than RW
    fig,ax=plt.subplots(figsize=(5.4,max(3.2,0.32*len(dm)))); y=np.arange(len(dm))
    colors=[BLUE if best.loc[m,pc]<0.05 else GREY for m in dm.index]
    ax.barh(y,dm.values,color=colors); ax.axvline(0,color="k",lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(dm.index,fontsize=8)
    ax.set_xlabel("−DM statistic (HAR vs random-walk vol; >0 favours HAR)")
    nsig=(best[pc]<0.05).sum()
    ax.set_title(f"Volatility is predictable ({nsig}/{len(dm)} markets, blue = p<0.05)",fontsize=10)
    savefig(fig,"fig4_manipulation")

  saved fig4_manipulation


## Cell 8 — Fig 5: residual RL vs the volatility-target rule (DSR reward)

In [8]:
if NB5B is None:
    print("skip fig5 — need NB5 armB (ext_results.csv or *_armB.csv)")
else:
    d=NB5B.copy()
    piv=d.pivot_table(index="market",columns=["agent","state"],values="Sharpe")
    rule_rows=d[(d.agent=="VTrule")].set_index("market")["Sharpe"]
    mk=list(rule_rows.index); x=np.arange(len(mk)); w=0.35
    fig,ax=plt.subplots(figsize=(6.0,3.8))
    for i,(ag,c) in enumerate([("PPO",BLUE),("Q-Learning",RED)]):
        cols=[cc for cc in piv.columns if cc[0]==ag and cc[1]=="Baseline"]
        if cols: ax.bar(x+(i-0.5)*w, piv[cols[0]].reindex(mk).values, w, label=ag, color=c, alpha=.85)
    ax.plot(x, rule_rows.reindex(mk).values, "k_", ms=20, mew=2.2, label="VT rule (bar to beat)")
    ax.set_xticks(x); ax.set_xticklabels(mk,rotation=45,ha="right",fontsize=8)
    ax.set_ylabel("test Sharpe"); ax.set_title("Residual RL vs volatility-target rule (DSR reward)",fontsize=10)
    ax.legend(frameon=False,fontsize=8); savefig(fig,"fig5_residual")
    # win counts
    for ag in ["PPO","Q-Learning"]:
        cols=[cc for cc in piv.columns if cc[0]==ag and cc[1]=="Baseline"]
        if cols:
            w_=(piv[cols[0]].reindex(mk).values>rule_rows.reindex(mk).values).sum()
            print(f"  {ag} Baseline beats VT rule in {w_}/{len(mk)} markets")

  saved fig5_residual
  PPO Baseline beats VT rule in 9/14 markets
  Q-Learning Baseline beats VT rule in 1/14 markets


## Cell 9 — list everything produced

In [9]:
import os
print("figures in",FIG,":")
for f in sorted(os.listdir(FIG)):
    if f.endswith(".pdf"): print("  ",f)
print("\nDownload the folder, or use it directly. PNGs are alongside for quick viewing.")

figures in /content/drive/MyDrive/nb4_outputs/figures_paper :
   fig0_design.pdf
   fig1_crossover.pdf
   fig2_forest.pdf
   fig3_accuracy_value.pdf
   fig4_manipulation.pdf
   fig5_residual.pdf
   fig7_stability.pdf

Download the folder, or use it directly. PNGs are alongside for quick viewing.
